# 4.9 · 支持向量回归 / Support Vector Regression (SVR)

> **课程定位 / Where this fits**
> 第 9 课，**Part 4 · 监督学习：回归**。
> Lesson 9, **Part 4 · Supervised Regression**.
>
> SVM(5.5)用"最大间隔"做分类。**SVR** 把同样的思想搬到回归：在预测线两侧划一条宽度 ε 的"管道"，**管道内的误差一律不罚**，只惩罚跑到管道外的点。配合**核技巧**，SVR 不用指定函数形式就能拟合任意非线性（对比 4.8 必须先写出函数）。
> SVM (5.5) uses max-margin for classification. **SVR** brings the same idea to regression: draw a tube of width ε around the prediction, **charge nothing for errors inside the tube**, penalize only points outside. With the **kernel trick**, SVR fits arbitrary nonlinearity without specifying a functional form (unlike 4.8, which needs one).
>
> 💼 **实战/面试视角**："SVR 和 SVM 关系 / ε 管道是什么 / C、gamma、epsilon 各管什么"。
> 💼 **Practical/interview angle:** "SVR vs SVM / what's the ε-tube / what do C, gamma, epsilon control".

> 💡 **面试相关 / Interview-relevant**
> - "ε-不敏感损失是什么 / SVR 怎么工作"（出镜率 ★★★★）
> - "C / gamma / epsilon 三个超参各控制什么"（★★★★★）
> - "为什么 SVR 必须缩放"（★★★★）
> - "支持向量在回归里是什么"（★★★）

---

## 学习目标 / Learning Objectives

1. 理解 **ε-不敏感损失**与"管道"思想。
   Understand the **ε-insensitive loss** and the "tube" idea.
2. 用**核 SVR** 拟合非线性，对比线性 SVR。
   Fit nonlinearity with kernel SVR vs linear SVR.
3. 看清回归里的**支持向量**（管道外/边界上的点）。
   See support vectors in regression (points on/outside the tube).
4. 调 **C / gamma / epsilon** 三个超参。
   Tune C / gamma / epsilon.
5. 牢记**缩放**和**排序数据要 shuffle 再 CV**。
   Remember scaling and shuffling sorted data before CV.

## 目录 / TOC
1. [先建直觉：ε 管道 ⭐](#1)
2. [📈 数据 + 线性 vs 核 SVR ⭐](#2)
3. [支持向量与管道 ⭐](#3)
4. [C/gamma/epsilon 调参 ⭐](#4)
5. [小结](#5)


<a id="1"></a>
## 1. 先建直觉：ε 管道 ⭐ / Intuition: The ε-Tube

普通回归（OLS）对**每一个**偏差都罚（平方误差），所以每个点都在"拉扯"那条线。SVR 换了个哲学：**只要预测和真实差距在 ε 之内，就认为"足够好"，零惩罚**。把这个 ε 容差画出来，就是预测线两侧的一条**宽度 2ε 的管道**。
Ordinary regression (OLS) penalizes **every** deviation (squared error), so every point tugs at the line. SVR takes a different philosophy: **as long as the prediction is within ε of the truth, it's "good enough", zero penalty**. Visualized, this ε tolerance is a **tube of width 2ε** around the prediction line.

这就是 **ε-不敏感损失**：$\max(0, |y-\hat y| - \varepsilon)$——管道内损失为 0，管道外才按距离线性罚。结果是：**只有落在管道边界上或管道外的点**才决定模型，它们就是回归里的**支持向量**；管道内的点完全无关紧要。这带来稀疏性和对小噪声的鲁棒性。
This is the **ε-insensitive loss**: $\max(0, |y-\hat y| - \varepsilon)$ — zero inside the tube, linearly penalized outside. The result: **only points on or outside the tube boundary** determine the model — these are the **support vectors** in regression; points inside the tube don't matter at all. This brings sparsity and robustness to small noise.


<a id="2"></a>
## 2. 数据 + 线性 vs 核 SVR ⭐ / Data & Linear vs Kernel SVR

用一个强非线性数据 $2\sin(x)$ + 噪声。和 SVM 一样，SVR 靠核技巧处理非线性：**linear 核只能画直线，RBF 核能拟合任意曲线**。SVR **对尺度敏感，必须先标准化**(3.4)。
A strongly nonlinear dataset $2\sin(x)$ + noise. Like SVM, SVR handles nonlinearity via the kernel trick: **the linear kernel only draws a line, the RBF kernel fits any curve**. SVR is **scale-sensitive — standardize first** (3.4).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

x = np.sort(rng.uniform(-4, 4, 200))
y = 2*np.sin(x) + rng.normal(0, 0.2, 200)      # 强非线性信号 + 噪声
X = x.reshape(-1, 1)
print("数据: 2·sin(x) + 噪声, 强非线性")

# SVR 对尺度敏感 → 标准化 / scale-sensitive
sc = StandardScaler().fit(X)
Xs = sc.transform(X)
x_plot = np.linspace(-4, 4, 300).reshape(-1, 1)
x_plot_s = sc.transform(x_plot)               # 用同一个 scaler 变换(防泄漏)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(x, y, alpha=0.4, s=15, label="数据 data")
for kernel, c in [("linear", "orange"), ("rbf", "red")]:
    svr = SVR(kernel=kernel, C=10, epsilon=0.1, gamma="scale").fit(Xs, y)
    ax.plot(x_plot, svr.predict(x_plot_s), color=c, lw=2, label=f"SVR ({kernel})")
ax.legend(); ax.set_title("线性 SVR 拟合不了 sin; RBF 核 SVR 完美捕捉非线性")
plt.tight_layout(); plt.show()
print("linear 核只能画直线; RBF 核自动拟合复杂非线性 — 核技巧威力(无需指定函数形式, 对比 4.8)")


<a id="3"></a>
## 3. 支持向量与管道 ⭐ / Support Vectors & the Tube

把 ε 管道和支持向量画出来：**只有红圈标出的支持向量**（落在管道边界或之外）才定义模型；管道内的灰点删掉也不影响结果。这正是 SVR 的稀疏性。
Visualizing the ε-tube and support vectors: **only the circled support vectors** (on or outside the tube) define the model; gray points inside the tube can be deleted with no effect. This is SVR's sparsity.


In [ ]:
svr = SVR(kernel="rbf", C=10, epsilon=0.2, gamma="scale").fit(Xs, y)
sv_idx = svr.support_                          # 支持向量在训练集里的索引
print(f"总样本 {len(X)}, 支持向量 {len(sv_idx)} 个 ({len(sv_idx)/len(X):.0%})")
print(f"→ 只有这 {len(sv_idx)} 个点定义整个模型, 其余(管道内)删了也不变\n")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(x, y, alpha=0.3, s=15, label="管道内点(无影响)")
ax.scatter(x[sv_idx], y[sv_idx], facecolor="none", edgecolor="red", s=60, label="支持向量 SVs")
pred_plot = svr.predict(x_plot_s)
ax.plot(x_plot, pred_plot, "b-", lw=2, label="SVR 预测")
# 画 ±ε 管道 / draw the ±epsilon tube
ax.fill_between(x_plot.ravel(), pred_plot-0.2, pred_plot+0.2, alpha=0.15, color="blue", label="ε 管道(±0.2)")
ax.legend(fontsize=8); ax.set_title("红圈=支持向量(管道边界/外); 管道内点对模型零贡献")
plt.tight_layout(); plt.show()


<a id="4"></a>
## 4. C/gamma/epsilon 调参 ⭐ / Tuning C/gamma/epsilon

SVR 有**三个**关键超参（面试要点）：
SVR has **three** key hyperparameters (interview point):
- **C**：管道外违反的惩罚强度（同 SVM）。大→拟合更紧但易过拟合，小→更平滑。
  **C:** penalty for violations outside the tube (as in SVM). Large → tighter fit but overfit-prone; small → smoother.
- **gamma**（RBF）：单点影响范围。大→曲线扭曲（过拟合），小→太平滑（欠拟合）。
  **gamma** (RBF): a point's reach. Large → wiggly (overfit); small → too smooth (underfit).
- **epsilon**：管道宽度。大→更多点落入管道、模型更稀疏更糙，小→更精细但支持向量更多。
  **epsilon:** tube width. Large → more points inside, sparser/coarser model; small → finer but more support vectors.

> ⚠️ 数据**按 x 排序**时，交叉验证**必须 shuffle**——否则连续切分会让验证折落在训练折的 x 范围之外，变成"外推"，CV 分数虚低（3.10 的教训）。
> ⚠️ When data is **sorted by x**, CV **must shuffle** — otherwise contiguous folds put validation outside the training x-range (extrapolation), deflating CV scores (lesson from 3.10).


In [ ]:
from sklearn.model_selection import GridSearchCV, KFold

# gamma 的影响: 欠拟合 → 刚好 → 过拟合 / gamma effect
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, g in zip(axes, [0.01, 0.5, 50]):
    s = SVR(kernel="rbf", C=10, gamma=g).fit(Xs, y)
    ax.scatter(x, y, alpha=0.3, s=12); ax.plot(x_plot, s.predict(x_plot_s), "r-", lw=2)
    ax.set_title(f"gamma={g}\n{'欠拟合(太平滑)' if g<0.1 else ('刚好' if g<5 else '过拟合(太扭曲)')}")
plt.tight_layout(); plt.show()

# 网格搜索三个超参; 数据按 x 排序 → KFold 必须 shuffle / grid search (shuffle!)
grid = GridSearchCV(SVR(kernel="rbf"),
                    {"C": [1, 10, 100], "gamma": [0.05, 0.2, 1, 5], "epsilon": [0.01, 0.05, 0.1]},
                    cv=KFold(5, shuffle=True, random_state=0), scoring="r2").fit(Xs, y)
print(f"最优超参 best: {grid.best_params_}")
print(f"最优 CV R² = {grid.best_score_:.3f}")
print("(数据按 x 排序, CV 必须 shuffle, 否则连续切分→外推→虚低, 3.10 教训)")


<a id="5"></a>
## 5. 小结 / Summary

```
SVR = SVM 思想用于回归: ε 管道内零惩罚, 只罚管道外(ε-不敏感损失 max(0,|y-ŷ|-ε))
支持向量 = 管道边界/外的点, 只有它们定义模型(稀疏); 管道内点无关
核技巧: RBF 核拟合任意非线性, 无需指定函数形式(对比 4.8 非线性回归要先写函数)
三超参: C(违反惩罚, 大→过拟合) / gamma(RBF 范围, 大→过拟合) / epsilon(管道宽度)
必须缩放(同 KNN/SVM); 排序数据 CV 必须 shuffle(3.10)
```

### 💡 面试速查 / Interview cheat-sheet
1. **ε-不敏感损失**: 管道内误差不罚, 只罚管道外 → 稀疏 + 抗小噪声。
   ε-insensitive loss: no penalty inside the tube → sparse + robust to small noise.
2. **三超参**: C(违反惩罚) / gamma(RBF 范围) / epsilon(管道宽度)。
   Three knobs: C / gamma / epsilon.
3. **核技巧拟合非线性**, 无需指定函数形式(SVR vs 4.8 的优势)。
   Kernel trick fits nonlinearity without a chosen function (SVR's edge over 4.8).
4. **必须缩放**; 排序数据 CV 要 shuffle。
   Must scale; shuffle sorted data before CV.
5. **大数据慢**(同 SVM, O(n²~n³)) — 大数据用树/线性模型。
   Slow on big data (like SVM) — use trees/linear models at scale.

### 下一节 / Next
**4.10 KNN 回归**——非参数惰性回归: 预测 = 最近 K 个邻居的目标均值。和 SVR/线性模型对比, 引出"局部 vs 全局"建模思路。
**4.10 KNN Regression** — nonparametric, lazy regression: predict = mean target of the K nearest neighbors. Contrasts local vs global modeling.
